In [1]:
import json
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import nltk
from nltk.corpus import stopwords
import umap
import plotly.express as px
import networkx as nx
from pyvis.network import Network
from sklearn.metrics import (
    adjusted_rand_score,
    precision_recall_fscore_support,
)

nltk.download("stopwords")

SEED = 72
DATA_DIR = Path("../data")

RESULTS_DIR = Path("../output/exp1_chapter4")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/navneetmann/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Theme clustering (blending Jaccard + BGE-M3 vectors)

In [ ]:
ARTICLES_CSV = DATA_DIR / "combined_2017_2023_themes_with_KG.csv"
SYNTHETIC_CSV = DATA_DIR / "synthetic_golden_set.csv"

articles = pd.read_csv(ARTICLES_CSV)
articles = articles[
    articles["themes"].notna() & articles["knowledge_graph"].notna()
].reset_index(drop=True)
articles["themes"] = articles["themes"].apply(json.loads)
articles["kg"] = articles["knowledge_graph"].apply(json.loads)

# paraphrased gold articles
synthetic = pd.read_csv(SYNTHETIC_CSV)
synthetic["themes"] = synthetic["themes"].apply(json.loads)

unique_themes = sorted(
    {entry["theme"] for lst in articles["themes"] for entry in lst}
    | {entry["theme"] for lst in synthetic["themes"] for entry in lst}
)
num_themes = len(unique_themes)
theme_index = {theme: i for i, theme in enumerate(unique_themes)}

print(f"articles with themes: {len(articles):,} | unique themes: {num_themes:,}")
articles[["title", "themes"]].head()

articles with themes: 5,433 | unique themes: 15,306


,title,themes
0,How Apple's secrecy can hurt consumers,"[{'theme': 'product update transparency', 'dim..."
1,GE to keep Rochester plant open,[{'theme': 'industrial manufacturing workforce...
2,Drones give North Dakota farmers a new tool to...,"[{'theme': 'precision agriculture adoption', '..."
3,Pogue's Basics: The secret Start menu in Windo...,[]
4,"All NFL games will air online, but watching wo...",[{'theme': 'sports media streaming fragmentati...


### Word overlaps on themes

In [ ]:
STOPWORDS = set(stopwords.words("english"))


def tokenize(theme):
    return {w for w in theme.lower().split() if w not in STOPWORDS}


theme_tokens = [tokenize(theme) for theme in unique_themes]

vocab = {}
theme_rows, token_cols = [], []
for i, tokens in enumerate(theme_tokens):
    for token in tokens:
        theme_rows.append(i)
        token_cols.append(vocab.setdefault(token, len(vocab)))

token_matrix = sp.csr_matrix(
    (np.ones(len(theme_rows), np.float32), (theme_rows, token_cols)),
    shape=(num_themes, len(vocab)),
)

# one sparse matmul gives every pairwise intersection; union follows by inclusion exclusion
intersection = (token_matrix @ token_matrix.T).toarray().astype(np.float32)
token_counts = np.asarray(token_matrix.sum(axis=1), np.float32).ravel()
union = token_counts[:, None] + token_counts[None, :]
union -= intersection

jacc_similarity = np.divide(intersection, union, out=intersection, where=union > 0)
np.fill_diagonal(jacc_similarity, 1)
del union, token_matrix


mean_offdiagonal = (jacc_similarity.sum() - np.trace(jacc_similarity)) / (
    num_themes * (num_themes - 1)
)
print(f"Unique themes : {num_themes:,} | token vocab: {len(vocab):,}")
print(f"Mean similarity (off-diagonal): {mean_offdiagonal:.4f}")

Unique themes : 15,306 | token vocab: 4,599
Mean similarity (off-diagonal): 0.0050


In [ ]:
JACC_THRESHOLD = 0.5


def preview_pairs(similarity, threshold):
    """Print the strongest and weakest theme pairs scoring at or above `threshold`."""

    left, right = np.where(similarity >= threshold)
    upper = left < right
    left, right = left[upper], right[upper]
    scores = similarity[left, right]
    ranked = np.argsort(-scores)

    print(f"Num of pairs: {len(ranked):,}\n")
    print("Some top pair examples:")
    for k in ranked[:10]:
        print(
            f"{unique_themes[left[k]]} -- {unique_themes[right[k]]} [{scores[k]:.3f}]"
        )
    print("\nSome bottom pair examples:")
    for k in ranked[-10:]:
        print(
            f"{unique_themes[left[k]]} -- {unique_themes[right[k]]} [{scores[k]:.3f}]"
        )


preview_pairs(jacc_similarity, JACC_THRESHOLD)

Num of pairs: 42,302

Some top pair examples:
startup venture capital funding -- venture capital funding startup [1.000]
manufacturing pandemic disruption -- pandemic disruption to manufacturing [1.000]
manufacturing robotics automation -- robotics automation for manufacturing [1.000]
market impact of pandemic -- pandemic impact on market [1.000]
market impact of pandemic -- pandemic market impact [1.000]
Electric Vehicle market competition -- electric vehicle market competition [1.000]
Analyst stock downgrade -- analyst stock downgrade [1.000]
Electric Vehicle market competition -- Electric vehicle market competition [1.000]
Cannabis market expansion -- cannabis market expansion [1.000]
Alternative investment management -- alternative investment management [1.000]

Some bottom pair examples:
cryptocurrency market surge -- cryptocurrency market turbulence [0.500]
cryptocurrency market surge -- cryptocurrency market trend [0.500]
cryptocurrency market speculation -- stock market specula

### Semantic similarity on themes

In [ ]:
EMB_CACHE = DATA_DIR / "theme_bge_m3.npz"

cached = {}
if EMB_CACHE.exists():
    archive = np.load(EMB_CACHE, allow_pickle=True)
    cached = dict(zip(archive["themes"].tolist(), archive["vecs"]))

missing = [theme for theme in unique_themes if theme not in cached]
if missing:
    encoder = SentenceTransformer("BAAI/bge-m3")
    new_vectors = encoder.encode(
        missing, batch_size=64, normalize_embeddings=True, show_progress_bar=True
    )
    cached.update(zip(missing, new_vectors.astype(np.float32)))
    np.savez_compressed(
        EMB_CACHE,
        themes=np.array(list(cached.keys()), dtype=object),
        vecs=np.stack(list(cached.values())).astype(np.float32),
    )
    print(f"encoded {len(missing)} new themes -> {EMB_CACHE}")


embeddings = np.stack([cached[theme] for theme in unique_themes]).astype(np.float32)
embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
print(f"Embeddings shape: {embeddings.shape}")

sem_similarity = (embeddings @ embeddings.T).astype(np.float32)
np.fill_diagonal(sem_similarity, 1)

mean_offdiagonal = (sem_similarity.sum() - np.trace(sem_similarity)) / (
    num_themes * (num_themes - 1)
)
print(f"Similarity matrix: {sem_similarity.shape}")
print(f"Mean similarity (off-diagonal): {mean_offdiagonal:.4f}")

Embeddings shape: (15306, 1024)


/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_31857/1540640748.py:26: RuntimeWarning: divide by zero encountered in matmul
  sem_similarity = (embeddings @ embeddings.T).astype(np.float32)
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_31857/1540640748.py:26: RuntimeWarning: overflow encountered in matmul
  sem_similarity = (embeddings @ embeddings.T).astype(np.float32)
/var/folders/g5/tqsgxfp922q6lw1z0fkx4k_00000gn/T/ipykernel_31857/1540640748.py:26: RuntimeWarning: invalid value encountered in matmul
  sem_similarity = (embeddings @ embeddings.T).astype(np.float32)


Similarity matrix: (15306, 15306)
Mean similarity (off-diagonal): 0.4041


In [6]:
SEM_THRESHOLD = 0.8

preview_pairs(sem_similarity, SEM_THRESHOLD)

Num of pairs: 26,519

Some top pair examples:
greenhouse gas emission reduction -- greenhouse gas emissions reduction [0.999]
Oil and gas drilling activity -- oil and gas drilling activity [0.997]
Subscription service growth -- subscription service growth [0.997]
Market expansion strategy -- market expansion strategy [0.996]
Price reduction strategy -- price reduction strategy [0.996]
biopharma executive leadership change -- biopharma leadership executive change [0.995]
Mergers and acquisitions activity -- mergers and acquisitions activity [0.995]
Social media user experience -- social media user experience [0.995]
holiday retail discount event -- retail holiday discount event [0.994]
Stock market rally -- stock market rally [0.994]

Some bottom pair examples:
pandemic driven economic disruption -- pandemic market disruption [0.800]
big tech dominance -- cloud computing dominance [0.800]
electric car market expansion -- electric vehicle growth [0.800]
China tech regulatory crackdown --

### Blend similarity + Cluster

In [ ]:
# Refer: https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html


def blended_similarity(alpha):
    """alpha * semantic + (1 - alpha) * Jaccard, float32 (alpha=1 -> pure semantic)."""

    combined = sem_similarity * np.float32(alpha)
    if alpha < 1.0:
        combined = combined + jacc_similarity * np.float32(1.0 - alpha)
    return combined


def condensed_distances(similarity):
    """1 - similarity, condensed to the upper triangle scipy's linkage expects."""

    distances = np.float32(1.0) - similarity
    np.clip(distances, 0.0, None, out=distances)
    # a handful of themes embed to a degenerate vector -> push them to max distance
    distances = np.nan_to_num(distances, nan=1.0, posinf=1.0, neginf=1.0)
    np.fill_diagonal(distances, 0.0)
    return squareform(distances, checks=False)


def cluster_themes(distances, tau, method):
    """Agglomerative clustering cut at distance tau -> theme -> 'c<label>'.

    Equivalent to AgglomerativeClustering(n_clusters=None, distance_threshold=tau,
    metric="precomputed", linkage=method), but builds the dendrogram once so it can
    be re-cut at other thresholds without refitting.
    """

    tree = linkage(distances, method=method)
    cluster_ids = fcluster(tree, t=tau, criterion="distance")
    theme_to_cluster = {
        theme: f"c{cid}" for theme, cid in zip(unique_themes, cluster_ids)
    }
    return theme_to_cluster, cluster_ids

In [ ]:
# ALPHA / TAU / LINKAGE were selected by sweeping on `val`; `test` is the number to report

ALPHA = 0.8  # blend: 1 = pure semantic, 0 = pure Jaccard
TAU = 0.325  # max cosine distance for a merge
LINKAGE = "average"

similarity = blended_similarity(ALPHA)
distances = condensed_distances(similarity)
del similarity

final_clusters, cluster_ids = cluster_themes(distances, TAU, LINKAGE)
del distances


cluster_sizes = np.bincount(cluster_ids)[1:]
print(f"clusters: {len(cluster_sizes):,}")
print(f"singletons: {(cluster_sizes == 1).sum():,} | largest: {cluster_sizes.max()}")

assignments = pd.DataFrame(
    {
        "theme": list(final_clusters.keys()),
        "canon_cluster": list(final_clusters.values()),
    }
)

assignments_path = RESULTS_DIR / "canon_agglomerative.csv"
assignments.to_csv(assignments_path, index=False)
print(
    f"wrote {len(assignments)} assignments "
    f"(alpha={ALPHA}, linkage={LINKAGE}, tau={TAU}) to {assignments_path}"
)

clusters: 7,563
singletons: 4,520 | largest: 59
wrote 15306 assignments (alpha=0.8, linkage=average, tau=0.325) to ../output/exp1_chapter4/canon_agglomerative.csv


### Evaluation on the golden set

In [ ]:
gold = pd.read_csv(DATA_DIR / "synthetic_gold_themes.csv")
gold = gold[gold["keep"] == 1]
n_gold_total = len(gold)

gold = gold[gold["theme"].isin(final_clusters)].reset_index(drop=True)
print(f"gold themes covered: {len(gold)} / {n_gold_total}")


split = pd.read_csv(DATA_DIR / "synthetic_gold_split.csv")[["theme", "split"]]
gold = gold.merge(split, on="theme", how="left")


def pairwise_scores(frame):
    if len(frame) < 2:
        return None
    gold_ids = pd.factorize(frame["gold_cluster_id"])[0]
    predicted_ids = pd.factorize(np.array([final_clusters[t] for t in frame["theme"]]))[
        0
    ]

    pair_index = np.triu_indices(len(frame), 1)
    same_gold = (gold_ids[:, None] == gold_ids[None, :])[pair_index]
    same_predicted = (predicted_ids[:, None] == predicted_ids[None, :])[pair_index]

    precision, recall, f1, _ = precision_recall_fscore_support(
        same_gold, same_predicted, average="binary", zero_division=0
    )
    return {
        "themes": len(frame),
        "clusters": frame["gold_cluster_id"].nunique(),
        "pairs": same_gold.size,
        "pos_pairs": int(same_gold.sum()),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "ARI": adjusted_rand_score(gold_ids, predicted_ids),
    }


by_split = {}
for name in ("train", "val", "test"):
    scored = pairwise_scores(gold[gold["split"] == name])
    if scored:
        by_split[name] = scored
by_split["whole gold set"] = pairwise_scores(gold)


results = pd.DataFrame(by_split).T
results.index.name = "split"
metrics_path = RESULTS_DIR / "exp1_metrics.csv"
results.to_csv(metrics_path)
print()
print(results.to_string(float_format=lambda x: f"{x:.3f}"))
print(f"\nwrote metrics -> {metrics_path}")


held_out = results.loc["test"]
print(
    f"\nHELD-OUT TEST — pairwise P/R/F1: "
    f"{held_out['precision']:.3f} / {held_out['recall']:.3f} / {held_out['f1']:.3f}"
    f"   ARI: {held_out['ARI']:.3f}   (n = {int(held_out['themes'])} themes)"
)

gold themes covered: 3133 / 3134


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)



                 themes  clusters       pairs  pos_pairs  precision  recall    f1   ARI
split                                                                                  
train          1854.000   600.000 1717731.000   2032.000      0.737   0.326 0.452 0.452
val             648.000   200.000  209628.000    854.000      0.842   0.317 0.461 0.460
test            628.000   201.000  196878.000    732.000      0.767   0.306 0.438 0.436
whole gold set 3133.000  1002.000 4906278.000   3621.000      0.523   0.320 0.397 0.397

wrote metrics -> ../output/exp1_chapter4/exp1_metrics.csv

HELD-OUT TEST — pairwise P/R/F1: 0.767 / 0.306 / 0.438   ARI: 0.436   (n = 628 themes)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)


In [30]:
originals = gold[gold["role"] == "original"]
original_themes = originals["theme"].to_numpy()
original_gold = originals["gold_cluster_id"].to_numpy()
original_pred = np.array([final_clusters[t] for t in original_themes])

breakdown = {}
for variant in ("syn1", "syn2", "syn3"):
    variants = gold[gold[f"in_{variant}"]]
    variant_themes = variants["theme"].to_numpy()
    variant_gold = variants["gold_cluster_id"].to_numpy()
    variant_pred = np.array([final_clusters[t] for t in variant_themes])

    same_gold = (original_gold[:, None] == variant_gold[None, :]).ravel()
    same_pred = (original_pred[:, None] == variant_pred[None, :]).ravel()
    # skip identical phrases (trivially in the same cluster)
    distinct = (original_themes[:, None] != variant_themes[None, :]).ravel()
    same_gold, same_pred = same_gold[distinct], same_pred[distinct]

    precision, recall, f1, _ = precision_recall_fscore_support(
        same_gold, same_pred, average="binary", zero_division=0
    )
    breakdown[f"parent-{variant}"] = {
        "n_pairs": same_gold.size,
        "n_pos": int(same_gold.sum()),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

print(pd.DataFrame(breakdown).T.round(3).to_string())

               n_pairs   n_pos  precision  recall     f1
parent-syn1  1080307.0  1097.0      0.710   0.844  0.771
parent-syn2  1095679.0  1133.0      0.438   0.074  0.127
parent-syn3  1092607.0  1130.0      0.432   0.073  0.126


### Inspect clusters

In [31]:
members_by_cluster = defaultdict(list)
for theme, cluster in final_clusters.items():
    members_by_cluster[cluster].append(theme)

sizes_desc = sorted((len(m) for m in members_by_cluster.values()), reverse=True)
print(f"{len(members_by_cluster)} clusters over {num_themes:,} themes")
print(f"Largest 10 cluster sizes: {sizes_desc[:10]}")
print(f"Singleton clusters (size = 1): {sizes_desc.count(1)}")

7563 clusters over 15,306 themes
Largest 10 cluster sizes: [59, 39, 33, 33, 27, 26, 26, 26, 26, 24]
Singleton clusters (size = 1): 4520


In [32]:
similarity = blended_similarity(ALPHA)


def member_similarities(members):
    """Blend similarities among a cluster's members, self-pairs masked out."""
    member_idx = [theme_index[m] for m in members]
    sims = similarity[np.ix_(member_idx, member_idx)].astype(np.float64)
    np.fill_diagonal(sims, np.nan)
    return sims


# representative theme per cluster = most central by the blend
cluster_representative = {}
for cluster, members in members_by_cluster.items():
    if len(members) == 1:
        cluster_representative[cluster] = members[0]
        continue
    centrality = np.nanmean(member_similarities(members), axis=1)
    cluster_representative[cluster] = members[int(np.nanargmax(centrality))]

largest_first = sorted(members_by_cluster.items(), key=lambda kv: -len(kv[1]))
for cluster, members in largest_first[:50]:
    cohesion = np.nanmean(member_similarities(members))
    print(f"Cluster {cluster} || {len(members)} themes || cohesion: {cohesion:.2f}")
    print(f"Representative: {cluster_representative[cluster]}")
    for member in members:
        print(f"- {member}")
    print("#############################")

del similarity

Cluster c4181 || 59 themes || cohesion: 0.71
Representative: central bank rate tightening
- Central Bank Interest Policy
- Central Bank Rate Tightening
- Central Bank Tightening
- Central bank interest policy
- Central bank monetary policy
- Central bank monetary tightening
- Central bank policy tightening
- Central bank rate tightening
- Central bank rates
- U.S. central bank policy
- central bank capital easing
- central bank easing policy
- central bank interest rate
- central bank interest rate hikes
- central bank interest rate policy
- central bank interest rates
- central bank lending rates
- central bank liquidity tightening
- central bank monetary easing
- central bank monetary policy
- central bank monetary tightening
- central bank money policy
- central bank policy decision
- central bank policy easing
- central bank policy response
- central bank policy tightening
- central bank quantitative easing
- central bank quantitative tightening
- central bank rate caution
- centra

In [ ]:
# save clusters in csv file

from collections import Counter

theme_to_cluster = final_clusters
SOURCE_CSV = ARTICLES_CSV

REPORT_DIR = RESULTS_DIR / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

JOIN = " | "
REPORT_COLS = [
    "article_id",
    "title",
    "url",
    "date_publish",
    "themes",
    "knowledge_graph",
]


def joined(values):
    """`JOIN`-join, neutralising any literal pipe inside a value so the separator
    stays unambiguous - 312 article titles in this corpus contain one."""
    return JOIN.join(str(v).replace("|", "/") for v in values)


article_rows, triple_rows = [], []
for chunk in pd.read_csv(SOURCE_CSV, usecols=REPORT_COLS, chunksize=20_000):
    chunk = chunk[chunk["themes"].notna() & chunk["knowledge_graph"].notna()]
    for row in chunk.itertuples(index=False):
        for entry in json.loads(row.themes):
            article_rows.append(
                (
                    entry["theme"],
                    entry.get("dimension"),
                    row.article_id,
                    row.title,
                    row.date_publish,
                    row.url,
                )
            )
        for record in json.loads(row.knowledge_graph):
            triplet = record.get("triplet", [])
            if len(triplet) != 5 or not record.get("theme"):
                continue
            head, head_type, relation, tail, tail_type = triplet
            triple_rows.append(
                (
                    record["theme"],
                    row.article_id,
                    head,
                    head_type,
                    relation,
                    tail,
                    tail_type,
                )
            )

theme_articles = pd.DataFrame(
    article_rows,
    columns=["theme", "dimension", "article_id", "title", "date_publish", "url"],
).drop_duplicates()
theme_triples = pd.DataFrame(
    triple_rows,
    columns=[
        "theme",
        "article_id",
        "head",
        "head_type",
        "relation",
        "tail",
        "tail_type",
    ],
)
print(
    f"corpus index: {len(theme_articles):,} theme-article pairs over "
    f"{theme_articles['theme'].nunique():,} themes | {len(theme_triples):,} triples"
)


_archive = np.load(DATA_DIR / "theme_bge_m3.npz", allow_pickle=True)
phrase_vectors = dict(zip(_archive["themes"].tolist(), _archive["vecs"]))

report_members = defaultdict(list)
for theme, cluster in theme_to_cluster.items():
    report_members[cluster].append(theme)

cluster_label, cluster_cohesion = {}, {}
for cluster, members in report_members.items():
    known = [m for m in members if m in phrase_vectors]
    if not known:
        cluster_label[cluster] = sorted(members)[0]
        cluster_cohesion[cluster] = float("nan")
        continue
    vectors = np.stack([phrase_vectors[m] for m in known]).astype(np.float32)
    vectors /= np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    centroid = vectors.mean(axis=0)
    centroid /= max(float(np.linalg.norm(centroid)), 1e-12)
    cluster_label[cluster] = known[int(np.argmax(vectors @ centroid))]
    if len(known) > 1:
        sims = vectors @ vectors.T
        cluster_cohesion[cluster] = float(sims[np.triu_indices(len(known), 1)].mean())
    else:
        cluster_cohesion[cluster] = float("nan")

assignments_long = pd.DataFrame(
    {
        "canon_cluster": list(theme_to_cluster.values()),
        "theme": list(theme_to_cluster),
    }
)
assignments_long["cluster_label"] = assignments_long["canon_cluster"].map(cluster_label)


article_map = assignments_long.merge(theme_articles, on="theme", how="left")
article_map["is_synthetic"] = article_map["article_id"].isna()
article_map = article_map[
    [
        "canon_cluster",
        "cluster_label",
        "theme",
        "dimension",
        "article_id",
        "title",
        "date_publish",
        "url",
        "is_synthetic",
    ]
].sort_values(["canon_cluster", "theme", "article_id"], na_position="last")
article_map.to_csv(REPORT_DIR / "cluster_theme_article_map.csv", index=False)

triple_map = assignments_long.merge(theme_triples, on="theme", how="inner")
triple_map = triple_map[
    [
        "canon_cluster",
        "cluster_label",
        "theme",
        "article_id",
        "head",
        "head_type",
        "relation",
        "tail",
        "tail_type",
    ]
].sort_values(["canon_cluster", "theme", "article_id"])
triple_map.to_csv(REPORT_DIR / "cluster_kg_triples.csv", index=False)

themes_with_articles = set(theme_articles["theme"])
real_rows = article_map[~article_map["is_synthetic"]]
articles_by_cluster = dict(tuple(real_rows.groupby("canon_cluster")))
triples_by_cluster = dict(tuple(triple_map.groupby("canon_cluster")))

summary_rows = []
for cluster, members in report_members.items():
    members = sorted(members)
    cluster_articles = articles_by_cluster.get(cluster)
    cluster_triples = triples_by_cluster.get(cluster)

    if cluster_articles is None:
        article_ids, titles, dimensions = [], [], Counter()
    else:
        article_ids = list(dict.fromkeys(cluster_articles["article_id"]))
        dimensions = Counter(cluster_articles["dimension"].dropna())

        ranked = cluster_articles.assign(
            _off_label=cluster_articles["theme"].ne(cluster_label[cluster]),
            _date=pd.to_datetime(cluster_articles["date_publish"], errors="coerce"),
        ).sort_values(["_off_label", "_date"], ascending=[True, False])
        titles = list(dict.fromkeys(ranked["title"].dropna()))

    entities = (
        Counter()
        if cluster_triples is None
        else Counter(zip(cluster_triples["tail"], cluster_triples["tail_type"]))
    )

    summary_rows.append(
        {
            "canon_cluster": cluster,
            "label": cluster_label[cluster],
            "n_themes": len(members),
            "n_articles": len(article_ids),
            "n_triples": 0 if cluster_triples is None else len(cluster_triples),
            "cohesion": cluster_cohesion[cluster],
            "dimensions": joined(f"{d}:{n}" for d, n in dimensions.most_common()),
            "member_themes": joined(members),
            "top_entities": joined(
                f"{entity} ({etype}) ×{n}"
                for (entity, etype), n in entities.most_common(10)
            ),
            "article_ids": joined(article_ids),
            "sample_titles": joined(titles[:5]),
            "n_synthetic_themes": sum(
                1 for m in members if m not in themes_with_articles
            ),
        }
    )

summary = pd.DataFrame(summary_rows).sort_values(
    ["n_themes", "canon_cluster"], ascending=[False, True]
)
summary.to_csv(REPORT_DIR / "cluster_summary.csv", index=False)

covered = sum(1 for t in theme_to_cluster if t in themes_with_articles)
print(
    f"report -> {REPORT_DIR}\n"
    f"  cluster_summary.csv            {len(summary):,} clusters\n"
    f"  cluster_theme_article_map.csv  {len(article_map):,} rows\n"
    f"  cluster_kg_triples.csv         {len(triple_map):,} rows\n"
    f"  {covered:,} / {len(theme_to_cluster):,} themes trace to a real article "
    f"({len(theme_to_cluster) - covered:,} synthetic)"
)

### UMAP overview of cluster centroids

In [33]:
# Refer: https://umap-learn.readthedocs.io/en/latest/how_umap_works.html

reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=SEED)
coords = reducer.fit_transform(embeddings)

centroid_rows = []
for cluster, members in members_by_cluster.items():
    member_coords = coords[[theme_index[m] for m in members]]
    centroid = member_coords.mean(axis=0)
    to_centroid = np.linalg.norm(member_coords - centroid, axis=1)
    centroid_rows.append(
        {
            "x": centroid[0],
            "y": centroid[1],
            "z": centroid[2],
            "cluster": str(cluster),
            "size": len(members),
            "representative": members[to_centroid.argmin()],
            "hover_themes": "<br>".join(members[:5]),
        }
    )

centroids = pd.DataFrame(centroid_rows)
centroids["size_display"] = centroids["size"] ** 1.5

MIN_CLUSTER_SIZE = 5
centroids = centroids[centroids["size"] >= MIN_CLUSTER_SIZE].reset_index(drop=True)
print(f"Showing remaining {len(centroids)} clusters")

fig = px.scatter_3d(
    centroids,
    x="x",
    y="y",
    z="z",
    size="size_display",
    color="cluster",
    hover_name="representative",
    hover_data={
        "hover_themes": True,
        "size": True,
        "size_display": False,
        "cluster": False,
        "x": False,
        "y": False,
        "z": False,
    },
    title="Theme Clusters [Single bubble/cluster]",
    opacity=0.8,
)
fig.update_traces(marker=dict(sizemode="area", sizeref=0.5, sizemin=10))
fig.show()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Showing remaining 604 clusters


### Visualize a cluster's theme KGs

In [ ]:
entity_graph = nx.MultiDiGraph()

for kg_list in articles["kg"]:
    for record in kg_list:
        triplet = record.get("triplet", [])
        theme = record.get("theme", "")
        if len(triplet) != 5 or not theme:
            continue
        head, head_type, relation, tail, tail_type = triplet
        for entity, entity_type in ((head, head_type), (tail, tail_type)):
            entity_graph.add_node(
                entity,
                entity_type=entity_type,
                is_theme=(str(entity_type).upper() == "CONCEPT"),
            )
        entity_graph.add_edge(head, tail, relation=relation, theme=theme)

print(
    f"Nodes: {entity_graph.number_of_nodes():,}, "
    f"Edges: {entity_graph.number_of_edges():,}"
)

Nodes: 45,471, Edges: 87,969


In [35]:
COLOR_MAP = {
    "COMP": "#f28e2b",
    "ORG": "#b07aa1",
    "ORG/GOV": "#ff9da7",
    "ORG/REG": "#fabfd2",
    "REG": "#fabfd2",
    "GPE": "#bab0ac",
    "PERSON": "#9c755f",
    "EVENT": "#edc948",
    "PRODUCT": "#d37295",
    "SECTOR": "#e15759",
    "CONCEPT": "#FF6347",  # theme anchors
    "ECON_INDICATOR": "#76b7b2",
    "MACRO_TREND": "#e377c2",
    "TECHNOLOGY": "#86bcb6",
    "TECHNOLOGY_CONCEPT": "#86bcb6",
    "POLICY": "#8cd17d",
    "POLICY_OR_REGULATION": "#8cd17d",
    "FIN_INSTRUMENT": "#59a14f",
    "CRYPTO": "#4e79a7",
}


def plot_kg(subgraph, output=None):
    output = output or RESULTS_DIR / "kg_view.html"
    network = Network(
        height="750px",
        width="100%",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )
    network.force_atlas_2based()
    for node, data in subgraph.nodes(data=True):
        entity_type = data.get("entity_type", "UNKNOWN")
        is_theme = data.get("is_theme", False)
        network.add_node(
            node,
            label=node,
            color=COLOR_MAP.get(entity_type, "#cccccc"),
            title=f"<b>{node}</b><br>Type: {entity_type}"
            + ("<br><i>theme anchor</i>" if is_theme else ""),
            size=30 if is_theme else 18,
            shape="diamond" if is_theme else "dot",
        )
    for head, tail, data in subgraph.edges(data=True):
        network.add_edge(
            head,
            tail,
            label=data.get("relation", ""),
            title=f"Rel: {data.get('relation','')}<br>Theme: {data.get('theme','')}",
            arrows="to",
            color="#555555",
        )
    network.show_buttons(filter_=["physics"])
    network.show(output)
    print(f"Saved -> {output}")

In [36]:
def viz_cluster_kg(cluster, output_dir=RESULTS_DIR):
    """Combined KG of every theme in a canonical cluster (CONCEPT theme hubs + entities)."""
    members = set(members_by_cluster[cluster])
    edges = [
        (head, tail, key)
        for head, tail, key, data in entity_graph.edges(data=True, keys=True)
        if data.get("theme") in members
    ]
    subgraph = entity_graph.edge_subgraph(edges)
    if subgraph.number_of_nodes() > 800:
        print(
            f"[warn] {subgraph.number_of_nodes():,} nodes — pyvis may be slow/cluttered"
        )

    output = f"{output_dir}/kg_cluster_{cluster}.html"
    plot_kg(subgraph, output)
    print(
        f"Cluster {cluster} | {len(members)} themes | "
        f"{subgraph.number_of_nodes():,} nodes | {subgraph.number_of_edges():,} edges | "
        f"rep: {cluster_representative.get(cluster)}"
    )
    return subgraph

In [37]:
viz_cluster_kg("c500")

../output/exp1_chapter4/kg_cluster_c500.html
Saved -> ../output/exp1_chapter4/kg_cluster_c500.html
Cluster c500 | 2 themes | 4 nodes | 4 edges | rep: adventure tourism industry expansion
